# Первый эксперимент

In [ ]:
import os
import math
import time
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as T
from torchvision.models import resnet18

try:
    from datasets import load_dataset
    DATASETS_AVAILABLE = True
except Exception:
    DATASETS_AVAILABLE = False


In [ ]:
@dataclass
class Config:
    epochs: int = 5
    max_samples: int = 300
    num_workers: int = 2
    img_size: int = 224
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    batch_grid: Tuple[int, ...] = (32, 64)
    lr_grid: Tuple[float, ...] = (1e-3, 3e-4)
    runs_per_opt: int = 10
    w_bits: int = 8
    a_bits: int = 8
    local_data_root: str = "/kaggle/input"

CFG = Config()

print("Device:", CFG.device)
print("Torch:", torch.__version__)


Device: cuda
Torch: 2.10.0+cu128


In [3]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = True
    cudnn.benchmark = False

set_seed(42)


In [ ]:
def build_transforms(img_size: int):
    train_tfms = T.Compose([
        T.Resize((img_size, img_size)),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])
    val_tfms = T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])
    return train_tfms, val_tfms


def subset_indices(n: int, max_samples: int) -> List[int]:
    if max_samples is None or max_samples <= 0 or n <= max_samples:
        return list(range(n))
    return list(range(max_samples))


def get_thudogs_datasets(cfg: Config):
    train_tfms, val_tfms = build_transforms(cfg.img_size)

    kaggle_root = "/kaggle/input/datasets/hadulc/tsinghua-dog-dataset/low-resolution"
    base_ds = torchvision.datasets.ImageFolder(kaggle_root)
    samples = base_ds.samples

    g = torch.Generator().manual_seed(42)
    indices = torch.randperm(len(samples), generator=g).tolist()
    split_idx = int(0.9 * len(indices))
    train_indices = indices[:split_idx]
    val_indices = indices[split_idx:]

    class ListDataset(torch.utils.data.Dataset):
        def __init__(self, samples, indices, transform):
            self.samples = samples
            self.indices = indices
            self.transform = transform

        def __len__(self):
            return len(self.indices)

        def __getitem__(self, idx):
            path, label = self.samples[self.indices[idx]]
            from PIL import Image
            img = Image.open(path).convert("RGB")
            return self.transform(img), label

    train_ds = ListDataset(samples, train_indices, train_tfms)
    val_ds = ListDataset(samples, val_indices, val_tfms)
    return train_ds, val_ds

In [ ]:
def build_loaders(cfg: Config, batch_size: int, train_ds=None, val_ds=None):
    if train_ds is None or val_ds is None:
        train_ds, val_ds = get_thudogs_datasets(cfg)
    train_indices = subset_indices(len(train_ds), cfg.max_samples)
    val_indices = subset_indices(len(val_ds), max(1, cfg.max_samples // 5))

    train_ds = Subset(train_ds, train_indices)
    val_ds = Subset(val_ds, val_indices)

    def collate(batch):
        if isinstance(batch[0], dict) and "pixel_values" in batch[0]:
            x = torch.stack([b["pixel_values"] for b in batch])
            y = torch.tensor([b["labels"] for b in batch])
            return x, y
        x, y = zip(*batch)
        return torch.stack(x), torch.tensor(y)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=True,
        collate_fn=collate,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
        collate_fn=collate,
    )
    return train_loader, val_loader


In [ ]:
class LSQFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, s, qn: int, qp: int):
        x_div = x / s
        x_clamped = x_div.clamp(qn, qp)
        q = x_clamped.round()
        ctx.save_for_backward(x_div, q, s)
        ctx.qn = qn
        ctx.qp = qp
        return q * s

    @staticmethod
    def backward(ctx, grad_output):
        x_div, q, s = ctx.saved_tensors
        qn = ctx.qn
        qp = ctx.qp
        indicator = (x_div >= qn) & (x_div <= qp)
        grad_x = grad_output * indicator
        g = 1.0 / math.sqrt(x_div.numel() * qp)
        grad_s = ((q - x_div) * grad_output).sum() * g
        return grad_x, grad_s, None, None


class LSQQuantizer(nn.Module):
    def __init__(self, bits: int, init_scale: float = None):
        super().__init__()
        self.bits = bits
        self.qn = -(2 ** (bits - 1))
        self.qp = (2 ** (bits - 1)) - 1
        if init_scale is None:
            self.s = nn.Parameter(torch.tensor(1.0))
            self._initialized = False
        else:
            self.s = nn.Parameter(torch.tensor(init_scale))
            self._initialized = True

    def init_from(self, x: torch.Tensor):
        with torch.no_grad():
            s = 2 * x.abs().mean() / math.sqrt(self.qp)
            self.s.copy_(s)
        self._initialized = True

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self._initialized:
            self.init_from(x)
        return LSQFunction.apply(x, self.s, self.qn, self.qp)


class LSQConv2d(nn.Module):
    def __init__(self, conv: nn.Conv2d, w_bits: int, a_bits: int):
        super().__init__()
        self.conv = conv
        self.w_q = LSQQuantizer(w_bits)
        self.a_q = LSQQuantizer(a_bits)

    def forward(self, x):
        x_q = self.a_q(x)
        w_q = self.w_q(self.conv.weight)
        return F.conv2d(
            x_q,
            w_q,
            self.conv.bias,
            stride=self.conv.stride,
            padding=self.conv.padding,
            dilation=self.conv.dilation,
            groups=self.conv.groups,
        )


class LSQLinear(nn.Module):
    def __init__(self, linear: nn.Linear, w_bits: int, a_bits: int):
        super().__init__()
        self.linear = linear
        self.w_q = LSQQuantizer(w_bits)
        self.a_q = LSQQuantizer(a_bits)

    def forward(self, x):
        x_q = self.a_q(x)
        w_q = self.w_q(self.linear.weight)
        return F.linear(x_q, w_q, self.linear.bias)


def quantize_resnet18_lsq(model: nn.Module, w_bits: int, a_bits: int) -> nn.Module:
    for name, module in model.named_children():
        if isinstance(module, nn.Conv2d):
            setattr(model, name, LSQConv2d(module, w_bits, a_bits))
        elif isinstance(module, nn.Linear):
            setattr(model, name, LSQLinear(module, w_bits, a_bits))
        else:
            quantize_resnet18_lsq(module, w_bits, a_bits)
    return model


In [ ]:
def build_model(num_classes: int, cfg: Config) -> nn.Module:
    weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
    model = resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model = quantize_resnet18_lsq(model, cfg.w_bits, cfg.a_bits)
    return model


In [ ]:
class SGLD(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, noise_scale=1.0, weight_decay=0.0):
        defaults = dict(lr=lr, noise_scale=noise_scale, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            loss = closure()
        for group in self.param_groups:
            lr = group["lr"]
            noise_scale = group["noise_scale"]
            weight_decay = group["weight_decay"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                grad = p.grad
                if weight_decay != 0:
                    grad = grad.add(p, alpha=weight_decay)
                noise = torch.randn_like(p) * math.sqrt(2.0 * lr * noise_scale)
                p.add_(grad, alpha=-lr)
                p.add_(noise)
        return loss


class SGHMC(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, alpha=0.1, noise_scale=1.0, weight_decay=0.0):
        defaults = dict(lr=lr, alpha=alpha, noise_scale=noise_scale, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            loss = closure()
        for group in self.param_groups:
            lr = group["lr"]
            alpha = group["alpha"]
            noise_scale = group["noise_scale"]
            weight_decay = group["weight_decay"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                grad = p.grad
                if weight_decay != 0:
                    grad = grad.add(p, alpha=weight_decay)
                state = self.state[p]
                if "momentum" not in state:
                    state["momentum"] = torch.zeros_like(p)
                v = state["momentum"]
                noise = torch.randn_like(p) * math.sqrt(2.0 * alpha * lr * noise_scale)
                v.mul_(1.0 - alpha).add_(grad, alpha=-lr).add_(noise)
                p.add_(v)
        return loss


def build_optimizer(name: str, params, lr: float):
    name = name.lower()
    if name == "sgd":
        return torch.optim.SGD(params, lr=lr, momentum=0.0)
    if name == "sgd_nesterov":
        return torch.optim.SGD(params, lr=lr, momentum=0.9, nesterov=True)
    if name == "adam":
        return torch.optim.Adam(params, lr=lr)
    if name == "adamw":
        return torch.optim.AdamW(params, lr=lr)
    if name == "rmsprop":
        return torch.optim.RMSprop(params, lr=lr)
    if name == "adagrad":
        return torch.optim.Adagrad(params, lr=lr)
    if name == "adadelta":
        return torch.optim.Adadelta(params, lr=lr)
    if name == "adamax":
        return torch.optim.Adamax(params, lr=lr)
    if name == "sgld":
        return SGLD(params, lr=lr, noise_scale=1.0)
    if name == "sghmc":
        return SGHMC(params, lr=lr, alpha=0.1, noise_scale=1.0)
    raise ValueError(f"Unknown optimizer: {name}")


OPTIMIZERS = [
    "sgd",
    "sgd_nesterov",
    "adam",
    "adamw",
    "rmsprop",
    "adagrad",
    "adadelta",
    "adamax",
    "sgld",
    "sghmc",
]


In [9]:
def accuracy_top1(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    loss_meter = 0.0
    acc_meter = 0.0
    n = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()
        bs = x.size(0)
        loss_meter += loss.item() * bs
        acc_meter += accuracy_top1(logits, y) * bs
        n += bs
    return loss_meter / n, acc_meter / n


def eval_one_epoch(model, loader, device):
    model.eval()
    loss_meter = 0.0
    acc_meter = 0.0
    n = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            bs = x.size(0)
            loss_meter += loss.item() * bs
            acc_meter += accuracy_top1(logits, y) * bs
            n += bs
    return loss_meter / n, acc_meter / n


In [ ]:
def get_num_classes(cfg: Config, train_ds=None) -> int:
    if train_ds is None:
        train_ds, _ = get_thudogs_datasets(cfg)
    if hasattr(train_ds, "features") and "label" in train_ds.features:
        return train_ds.features["label"].num_classes
    if hasattr(train_ds, "classes"):
        return len(train_ds.classes)
    if hasattr(train_ds, "dataset") and hasattr(train_ds.dataset, "classes"):
        return len(train_ds.dataset.classes)
    return 130


In [ ]:
def run_experiments(cfg: Config) -> pd.DataFrame:
    results = []
    train_ds, val_ds = get_thudogs_datasets(cfg)
    num_classes = get_num_classes(cfg, train_ds=train_ds)

    loader_cache = {}
    for bs in cfg.batch_grid:
        loader_cache[bs] = build_loaders(cfg, batch_size=bs, train_ds=train_ds, val_ds=val_ds)

    grid = [(bs, lr) for bs in cfg.batch_grid for lr in cfg.lr_grid]

    for opt_name in OPTIMIZERS:
        for run_idx in range(cfg.runs_per_opt):
            bs, lr = grid[run_idx % len(grid)]
            seed = 1000 * OPTIMIZERS.index(opt_name) + run_idx
            set_seed(seed)

            train_loader, val_loader = loader_cache[bs]
            model = build_model(num_classes, cfg).to(cfg.device)
            optimizer = build_optimizer(opt_name, model.parameters(), lr=lr)

            start = time.time()
            for _ in range(cfg.epochs):
                train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, cfg.device)
                val_loss, val_acc = eval_one_epoch(model, val_loader, cfg.device)
            elapsed = time.time() - start

            results.append({
                "optimizer": opt_name,
                "run": run_idx,
                "seed": seed,
                "batch_size": bs,
                "lr": lr,
                "epochs": cfg.epochs,
                "max_samples": cfg.max_samples,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "time_sec": elapsed,
            })
            print(f"{opt_name} | run {run_idx} | bs={bs} lr={lr} | val_acc={val_acc:.4f} | {elapsed:.1f}s")

    return pd.DataFrame(results)


In [ ]:
RUN_ALL = True

if RUN_ALL:
    df = run_experiments(CFG)
    df.to_csv('/kaggle/working/my_dataset.csv', index=False)
    df.head()


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 172MB/s] 


sgd | run 0 | bs=32 lr=0.001 | val_acc=0.1333 | 11.1s
sgd | run 1 | bs=32 lr=0.0003 | val_acc=0.0667 | 9.3s
sgd | run 2 | bs=64 lr=0.001 | val_acc=0.0500 | 10.9s
sgd | run 3 | bs=64 lr=0.0003 | val_acc=0.0167 | 11.0s
sgd | run 4 | bs=32 lr=0.001 | val_acc=0.0667 | 9.7s
sgd | run 5 | bs=32 lr=0.0003 | val_acc=0.0000 | 9.6s
sgd | run 6 | bs=64 lr=0.001 | val_acc=0.0500 | 10.8s
sgd | run 7 | bs=64 lr=0.0003 | val_acc=0.0000 | 10.8s
sgd | run 8 | bs=32 lr=0.001 | val_acc=0.0667 | 9.6s
sgd | run 9 | bs=32 lr=0.0003 | val_acc=0.0333 | 9.6s
sgd_nesterov | run 0 | bs=32 lr=0.001 | val_acc=0.4000 | 9.6s
sgd_nesterov | run 1 | bs=32 lr=0.0003 | val_acc=0.2167 | 9.8s
sgd_nesterov | run 2 | bs=64 lr=0.001 | val_acc=0.3000 | 11.2s
sgd_nesterov | run 3 | bs=64 lr=0.0003 | val_acc=0.1333 | 11.2s
sgd_nesterov | run 4 | bs=32 lr=0.001 | val_acc=0.4000 | 10.0s
sgd_nesterov | run 5 | bs=32 lr=0.0003 | val_acc=0.2333 | 10.1s
sgd_nesterov | run 6 | bs=64 lr=0.001 | val_acc=0.2667 | 11.3s
sgd_nesterov | run

In [13]:
def summarize_results(df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        df.groupby("optimizer")
        .agg(
            val_acc_mean=("val_acc", "mean"),
            val_acc_std=("val_acc", "std"),
            time_mean_sec=("time_sec", "mean"),
            time_std_sec=("time_sec", "std"),
        )
        .sort_values("val_acc_mean", ascending=False)
    )
    return summary

if "df" in globals():
    summary = summarize_results(df)
    display(summary)


,val_acc_mean,val_acc_std,time_mean_sec,time_std_sec
optimizer,,,,
adagrad,0.368333,0.101059,10.513662,0.578584
sgd_nesterov,0.263333,0.109375,10.478909,0.685118
adamax,0.248333,0.161102,10.487976,0.579927
adam,0.138333,0.067151,10.540288,0.653153
adamw,0.131667,0.073472,10.662180,0.624199
rmsprop,0.076667,0.055667,10.562821,0.625650
sgd,0.048333,0.039636,10.238061,0.730437
sgld,0.020000,0.041425,10.458129,0.567981
adadelta,0.016667,0.015713,10.580757,0.544978
